# Qwen receipt attack results: validation review

This notebook reviews Sakeeth's completed five-worker Qwen receipt-total attack run without copying or modifying his output files. On CARC it reads the shared run directly from `/project2/yzhao010_1531/qwen-receipt-runs/qwen_attack/qwen_receipt_validation_v1`.

It provides a completion audit, results by dataset and perturbation budget, and native-resolution clean/attacked/difference views. A merged output directory is not required.

In [1]:
from pathlib import Path
import base64
import io
import json
import os
import sys

import ipywidgets as widgets
import matplotlib
import numpy as np
import pandas as pd
from IPython.display import HTML, Markdown, clear_output, display
from PIL import Image

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if not (ROOT / 'src').is_dir():
    raise RuntimeError('Open this notebook from the poisoned-paperwork repository root or notebooks/ directory.')

sys.path.insert(0, str(ROOT / 'src'))
from data import load_processed_rows, row_images

RUN_ID = 'qwen_receipt_validation_v1'
CARC_RUN_DIR = Path('/project2/yzhao010_1531/qwen-receipt-runs/qwen_attack') / RUN_ID
LOCAL_RUN_DIR = ROOT / 'outputs/qwen_attack' / RUN_ID
RUN_DIR = Path(os.environ.get('PP_RECEIPT_REVIEW_RUN_DIR', CARC_RUN_DIR))
if not RUN_DIR.is_dir() and LOCAL_RUN_DIR.is_dir():
    RUN_DIR = LOCAL_RUN_DIR
if not RUN_DIR.is_dir():
    raise FileNotFoundError(
        f'Receipt run not found at {RUN_DIR}. Set PP_RECEIPT_REVIEW_RUN_DIR to a copied run directory.'
    )

config = json.loads((RUN_DIR / 'config.json').read_text())
record_paths = sorted(RUN_DIR.glob('shards/worker_*/records/*.json'))
records = []
for path in record_paths:
    record = json.loads(path.read_text())
    record['_record_path'] = str(path)
    record['_worker'] = path.parents[1].name
    records.append(record)
if not records:
    raise FileNotFoundError(f'No receipt records found below {RUN_DIR}')

def first_value(record, *keys, default=None):
    for key in keys:
        value = record.get(key)
        if value is not None:
            return value
    return default

def record_id(record):
    return str(record['record_id'])

def epsilon_of(record):
    return int(record['epsilon_255'])

def succeeded(record):
    return bool(first_value(record, 'targeted_success_reloaded', 'saved_image_success', default=False))

print(f'Loaded {len(records)} records from {RUN_DIR}')

Loaded 876 records from /project2/yzhao010_1531/qwen-receipt-runs/qwen_attack/qwen_receipt_validation_v1


In [2]:
epsilons = [int(value) for value in config['attack']['epsilons_255']]
expected_by_dataset = {key: int(value) for key, value in config['expected_validation_candidates'].items()}
expected_documents = sum(expected_by_dataset.values())
expected_records = expected_documents * len(epsilons)
unique_documents = sorted({record_id(record) for record in records})
complete_markers = sorted(RUN_DIR.glob('shards/worker_*/WORKER_COMPLETE.json'))
failure_paths = sorted(RUN_DIR.glob('shards/worker_*/failures/*.json'))
keys = [(record_id(record), epsilon_of(record)) for record in records]
duplicate_keys = len(keys) - len(set(keys))

rows = []
for dataset in config['datasets']:
    for epsilon in epsilons:
        group = [
            record for record in records
            if record.get('dataset_name') == dataset and epsilon_of(record) == epsilon
        ]
        successes = sum(succeeded(record) for record in group)
        def metric(*names):
            return [float(first_value(record, *names)) for record in group if first_value(record, *names) is not None]
        rows.append({
            'dataset': dataset,
            'epsilon': epsilon,
            'records': len(group),
            'targeted successes': successes,
            'conditional ASR': successes / len(group) if group else np.nan,
            'median LPIPS': np.median(metric('lpips')) if metric('lpips') else np.nan,
            'median SSIM': np.median(metric('ssim')) if metric('ssim') else np.nan,
            'median runtime (s)': np.median(metric('runtime_seconds', 'attack_runtime_seconds')) if metric('runtime_seconds', 'attack_runtime_seconds') else np.nan,
            'median steps': np.median(metric('completed_steps', 'selected_step')) if metric('completed_steps', 'selected_step') else np.nan,
        })
summary = pd.DataFrame(rows).set_index(['dataset', 'epsilon'])
display(summary.style.format({
    'conditional ASR': '{:.1%}',
    'median LPIPS': '{:.3f}',
    'median SSIM': '{:.3f}',
    'median runtime (s)': '{:.1f}',
    'median steps': '{:.1f}',
}))

worker_counts = pd.Series([record['_worker'] for record in records]).value_counts().sort_index()
dataset_counts = pd.Series([record.get('dataset_name') for record in records]).value_counts().sort_index()
audit = pd.Series({
    'records': f'{len(records)}/{expected_records}',
    'unique receipts': f'{len(unique_documents)}/{expected_documents}',
    'targeted successes': f'{sum(succeeded(record) for record in records)}/{len(records)}',
    'worker completion markers': f'{len(complete_markers)}/5',
    'historical failure files': len(failure_paths),
    'duplicate receipt/epsilon keys': duplicate_keys,
}, name='completion audit')
display(audit.to_frame())
display(pd.concat([worker_counts.rename('records by worker'), dataset_counts.rename('records by dataset')], axis=1))

audit_passed = (
    len(records) == expected_records
    and len(unique_documents) == expected_documents
    and len(complete_markers) == 5
    and duplicate_keys == 0
)
if audit_passed:
    display(Markdown('**Completion audit passed:** all 876 expected experiments are present across all five workers.'))
else:
    display(Markdown('**Completion audit is incomplete:** do not report final values until the missing or duplicate records are resolved.'))

,completion audit
records,876/876
unique receipts,219/219
targeted successes,471/876
worker completion markers,5/5
historical failure files,0
duplicate receipt/epsilon keys,0


,records by worker,records by dataset
worker_00,176.0,NaN
worker_01,176.0,NaN
worker_02,176.0,NaN
worker_03,176.0,NaN
worker_04,172.0,NaN
cord_v2,NaN,384.0
sroie,NaN,492.0


**Completion audit passed:** all 876 expected experiments are present across all five workers.

## Native-resolution receipt viewer

Choose a dataset, ε, and attack outcome, then render one receipt. Every panel uses the original PNG dimensions and lossless encoding.

- **Magnitude heatmap** uses the same fixed 0-to-ε color scale for every receipt at that ε.
- **Absolute ×N** amplifies RGB absolute differences for diagnosis.
- **Signed residual** maps no change to mid-gray and preserves the direction of each RGB change.
- **A/B toggle** alternates clean and attacked pixels at identical coordinates, which is the best way to check for visible flicker.

In [ ]:
rows_by_id = load_processed_rows(
    ROOT, split=config.get('split', 'validation'), record_ids=set(unique_documents)
)

def png_bytes(image):
    buffer = io.BytesIO()
    image.save(buffer, format='PNG', compress_level=1)
    return buffer.getvalue()

def png_data_uri(image):
    return 'data:image/png;base64,' + base64.b64encode(png_bytes(image)).decode('ascii')

def clean_image(record):
    pages = row_images(rows_by_id[record_id(record)])
    if len(pages) != 1:
        raise ValueError(f'Expected one receipt page for {record_id(record)}; found {len(pages)}')
    return pages[0].convert('RGB')

def attacked_image_path(record):
    metadata = record.get('images') or []
    if not metadata or not isinstance(metadata[0], dict) or not metadata[0].get('path'):
        raise ValueError(f'Record has no attacked-image metadata: {record_id(record)}')
    raw = Path(metadata[0]['path'])
    candidates = [raw]
    if not raw.is_absolute():
        candidates.extend([RUN_DIR / raw, Path(record['_record_path']).parent / raw])
    candidates.append(Path(record['_record_path']).parents[1] / 'images' / Path(record['_record_path']).stem / 'page_000.png')
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(f'Attacked PNG is missing or unreadable for {record_id(record)}')

def diagnostic_image(clean, attacked, kind, multiplier, epsilon):
    clean_array = np.asarray(clean, dtype=np.int16)
    attacked_array = np.asarray(attacked, dtype=np.int16)
    delta = attacked_array - clean_array
    if kind == 'Absolute ×N':
        shown = np.clip(np.abs(delta) * multiplier, 0, 255).astype(np.uint8)
        return Image.fromarray(shown), f'Absolute difference ×{multiplier}'
    if kind == 'Signed residual':
        scale = max(float(epsilon), 1.0)
        shown = np.clip(127.5 + delta * (127.5 / scale), 0, 255).astype(np.uint8)
        return Image.fromarray(shown), f'Signed residual, fixed range ±{epsilon}'
    magnitude = np.max(np.abs(delta), axis=2) / max(float(epsilon), 1.0)
    colored = matplotlib.colormaps['magma'](np.clip(magnitude, 0, 1))[:, :, :3]
    return Image.fromarray((colored * 255).astype(np.uint8)), f'Magnitude heatmap, fixed range 0–{epsilon}'

dataset_widget = widgets.Dropdown(options=config['datasets'], description='Dataset:')
epsilon_widget = widgets.Dropdown(options=epsilons, value=max(epsilons), description='ε:')
outcome_widget = widgets.Dropdown(options=['All', 'Successful', 'Unsuccessful'], description='Outcome:')
record_widget = widgets.Dropdown(description='Receipt:', layout=widgets.Layout(width='850px'))
difference_widget = widgets.Dropdown(
    options=['Magnitude heatmap', 'Absolute ×N', 'Signed residual'],
    value='Magnitude heatmap', description='Difference:'
)
multiplier_widget = widgets.IntSlider(value=16, min=1, max=64, step=1, description='Multiplier:')
render_button = widgets.Button(description='Render native resolution', button_style='primary')
viewer_output = widgets.Output()
ab_output = widgets.Output()
ab_toggle = widgets.ToggleButtons(options=[('Clean', 'clean'), ('Attacked', 'attacked')], description='A/B:')
viewer_state = {}

def record_label(index):
    record = records[index]
    status = 'SUCCESS' if succeeded(record) else 'failure'
    clean = first_value(record, 'clean_target', 'clean_prediction', default='?')
    target = first_value(record, 'attack_target', 'target_text', default='?')
    prediction = first_value(record, 'prediction_normalized', 'final_prediction_normalized', default='?')
    return f'{record_id(record)} | {status} | {clean} → {target} | output={prediction}'

def refresh_records(*_):
    choices = []
    for index, record in enumerate(records):
        if record.get('dataset_name') != dataset_widget.value or epsilon_of(record) != epsilon_widget.value:
            continue
        if outcome_widget.value == 'Successful' and not succeeded(record):
            continue
        if outcome_widget.value == 'Unsuccessful' and succeeded(record):
            continue
        choices.append((record_label(index), index))
    record_widget.options = choices
    if choices:
        record_widget.value = choices[0][1]

def render_ab():
    if not viewer_state:
        return
    image = viewer_state[ab_toggle.value]
    with ab_output:
        clear_output(wait=True)
        display(HTML(
            '<div style="max-width:100%;overflow:auto;border:1px solid #777;background:#222;padding:8px">'
            f'<img src="{png_data_uri(image)}" width="{image.width}" height="{image.height}" '
            'style="display:block;max-width:none"></div>'
        ))

def render_selected(_=None):
    with viewer_output:
        clear_output(wait=True)
        if record_widget.value is None:
            display(Markdown('No records match this filter.'))
            return
        record = records[record_widget.value]
        clean = clean_image(record)
        attacked = Image.open(attacked_image_path(record)).convert('RGB')
        if clean.size != attacked.size:
            raise ValueError(f'Dimension mismatch: clean={clean.size}, attacked={attacked.size}')
        epsilon = epsilon_of(record)
        diagnostic, diagnostic_title = diagnostic_image(
            clean, attacked, difference_widget.value, multiplier_widget.value, epsilon
        )
        meta = pd.Series({
            'record': record_id(record),
            'dataset': record.get('dataset_name'),
            'worker': record['_worker'],
            'epsilon': epsilon,
            'targeted success': succeeded(record),
            'clean total': first_value(record, 'clean_target', 'clean_prediction'),
            'attack target (10×)': first_value(record, 'attack_target', 'target_text'),
            'reloaded output': first_value(record, 'prediction_normalized', 'final_prediction_normalized'),
            'dimensions': f'{clean.width}×{clean.height}',
            'L∞ (255 scale)': first_value(record, 'linf_255'),
            'LPIPS': first_value(record, 'lpips'),
            'SSIM': first_value(record, 'ssim'),
            'runtime (s)': first_value(record, 'runtime_seconds', 'attack_runtime_seconds'),
            'completed steps': first_value(record, 'completed_steps', 'selected_step'),
        }, name='selected experiment')
        display(meta.to_frame())
        cards = []
        for title, image in [('Clean receipt', clean), (f'Attacked ε={epsilon}', attacked), (diagnostic_title, diagnostic)]:
            cards.append(
                '<div style="display:inline-block;vertical-align:top;margin-right:16px">'
                f'<h3 style="margin:0 0 6px 0">{title}</h3>'
                f'<img src="{png_data_uri(image)}" width="{image.width}" height="{image.height}" '
                'style="display:block;max-width:none"></div>'
            )
        display(HTML(
            '<div style="width:100%;overflow:auto;white-space:nowrap;border:1px solid #777;background:#222;padding:10px">'
            + ''.join(cards) + '</div>'
        ))
        viewer_state.clear()
        viewer_state.update(clean=clean, attacked=attacked)
        display(Markdown('### Exact-alignment A/B toggle'))
        display(ab_toggle, ab_output)
        render_ab()

dataset_widget.observe(refresh_records, names='value')
epsilon_widget.observe(refresh_records, names='value')
outcome_widget.observe(refresh_records, names='value')
ab_toggle.observe(lambda change: render_ab(), names='value')
render_button.on_click(render_selected)
refresh_records()
display(widgets.VBox([
    widgets.HBox([dataset_widget, epsilon_widget, outcome_widget]),
    record_widget,
    widgets.HBox([difference_widget, multiplier_widget]),
    render_button,
    viewer_output,
]))

## Interpretation

A targeted success means the saved adversarial receipt PNG was reloaded and Qwen returned the exact requested 10× target total. The heatmap and residual panels intentionally exaggerate pixel changes and are not model inputs.

Report conditional ASR separately for SROIE and CORD at every ε, alongside LPIPS, SSIM, and representative native-resolution successes and failures. A visually subtle A/B comparison supports—but does not by itself prove—imperceptibility.